In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "HappyDoggo1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'gs_logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

#Clean, simple header with logo, title & subtitle
app.layout = html.Div([
    html.Div(
        children=[
            html.A(
                html.Img(
                    src='data:image/png;base64,{}'.format(encoded_image.decode()),
                    style={'height': '90px', 'marginRight': '24px'}
                ),
                # URL link to SNHU website (blank page target)
                href='https://www.snhu.edu',
                target='_blank'
            ),
            html.Div([
                html.H1('Grazioso Salvare Rescue Dashboard'),
                html.H3('CS-340 Project Two | Michael Gresh')
            ])
        ],
        #CSS styling
        style={
            'display': 'flex',
            'alignItems': 'center',
            'justifyContent': 'center',
            'padding': '10px 10px',
            'marginBottom': '10px',
            'backgroundColor': '#ffffff',
        }
    ),
    html.Hr(),

    html.Div([
        # FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': ' Water Rescue', 'value': 'Water Rescue'},
                {'label': ' Mountain or Wilderness Rescue', 'value': 'Mountain or Wilderness Rescue'},
                {'label': ' Disaster or Individual Tracking', 'value': 'Disaster or Individual Tracking'},
                {'label': ' Reset', 'value': 'Reset'}
            ],
            # Set the default value to 'Reset'
            value='Reset',
            labelStyle={'display': 'inline-block', 'margin-right': '20px'}
        )
    ]),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),

        # FIXME: Set up the features for your interactive data table to make it user-friendly for your client
        # If you completed the Module Six Assignment, you can copy in the code you created here
        row_selectable="single",
        selected_rows=[0],
        selected_columns=[],
        page_size=10,
        sort_action="native",
        filter_action="native",
        column_selectable="single",
        style_table={
            'overflowX': 'auto',
            'maxHeight': '300px',
            'overflowY': 'auto'
        },
        style_cell={
            'textAlign': 'left',
            'minWidth': '100px',
            'width': '150px',
            'maxWidth': '200px',
            'whiteSpace': 'normal'
        },
        style_header={
            'fontWeight': 'bold',
            'backgroundColor': 'rgb(230, 230, 230)'
        }
    ),

    html.Br(),
    html.Hr(),

    # This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
            )
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    ## FIX ME Add code to filter interactive data table with MongoDB queries
    # Water Rescue criteria
    if filter_type == 'Water Rescue':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$regex": "Labrador Retriever|Chesa Bay Retr|Newfoundland",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    # Mountain or Wilderness Rescue criteria
    elif filter_type == 'Mountain or Wilderness Rescue':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$regex": "German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    # Disaster or Individual Tracking criteria
    elif filter_type == 'Disaster or Individual Tracking':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$regex": "Doberman Pinsch|German Shepherd|Golden Retriever|Bloodhound|Rottweiler",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    else:
        query = {}

    # execute query and convert into dataframe for display in datatable
    filtered_df = pd.DataFrame.from_records(db.read(query))

    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)

    return filtered_df.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('filter-type', 'value')] # added filter_type to update graph title based on filter
)
def update_graphs(viewData, filter_type):
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #
    if viewData is None or len(viewData) == 0:
        return [
            html.Div("No data available for the selected filter.")
        ]
        
    # Convert the viewData to a DataFrame for plotting
    dff = pd.DataFrame.from_dict(viewData)

    # Calculate the breed counts for the top 10 breeds
    breed_counts = (
        dff['breed']
        .value_counts()
        .nlargest(10)
        .reset_index()
    )

    # Rename the columns for better readability in the plot
    breed_counts.columns = ['breed', 'count']

    # Define a title lookup dictionary to map filter types to chart titles
    title_lookup = {
        'Water Rescue': 'Top Water Rescue Candidate Breeds',
        'Mountain or Wilderness Rescue': 'Top Mountain or Wilderness Rescue Candidate Breeds',
        'Disaster or Individual Tracking': 'Top Disaster or Tracking Candidate Breeds',
        'Reset': 'Top Rescue Candidate Breeds'
    }

    # Get the appropriate chart title based on the selected filter type
    chart_title = title_lookup.get(filter_type, 'Top Rescue Candidate Breeds')

    # Create a horizontal bar chart using Plotly Express
    fig = px.bar(
        breed_counts,
        x='count',
        y='breed',
        orientation='h',
        title=chart_title
    )

    # Update the layout of the figure to improve appearance
    fig.update_layout(
        yaxis={'categoryorder': 'total ascending'},
        margin=dict(l=10, r=10, t=50, b=40),
        height=500
    )

    return [
        dcc.Graph(figure=fig)
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    # If no columns are selected, return an empty list to avoid errors
    if selected_columns is None:
        selected_columns = []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    # Added check to ensure viewData is not None and contains data before map creation to prevent filter errors
    if viewData is None or len(viewData) == 0:
        return [
            html.Div("No map data available for the selected filter.")
        ]

    dff = pd.DataFrame.from_dict(viewData)

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]
    # The map will be centered on the location of the animal in the selected row of the datatable
    selected_lat = dff.iloc[row]['location_lat']
    selected_lon = dff.iloc[row]['location_long']

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[selected_lat, selected_lon],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[selected_lat, selected_lon],
                    children=[
                        dl.Tooltip(dff.iloc[row]['breed']),
                        dl.Popup([
                            # Revised animal detail tooltip to include name, breed, sex, age, and outcome type 
                            html.H3("Animal Details"),
                            html.P("Name: {}".format(dff.iloc[row]['name'])),
                            html.P("Breed: {}".format(dff.iloc[row]['breed'])),
                            html.P("Sex: {}".format(dff.iloc[row]['sex_upon_outcome'])),
                            html.P("Age: {}".format(dff.iloc[row]['age_upon_outcome'])),
                            html.P("Outcome: {}".format(dff.iloc[row]['outcome_type']))
                        ])
                    ]
                )
            ]
        )
    ]

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://stellagenesis-microgreek-3000.codio.io/proxy/8050/
